# backgound 5000 step, 5 variables, 12 walkers

In [ ]:
#! /usr/bin/env python3
import numpy as np
import emcee
import sys, os
import matplotlib.pyplot as plt
import h5py
from scipy.stats import norm

In [ ]:
h5=h5py.File(f'run_juno_emcee_pseudo_H2O_T_RHmax_parallel_step_10000.h5', 'r') 

chain = h5['mcmc']['chain'][:,:,:]
[nstep,nwalk,ndim]=chain.shape
print([nstep,nwalk,ndim])
h5.close()
labels=["qH2O [ppm]", "Temperature [K]", "RH_max_NH3"]

## inspect step

In [ ]:
fig, ax = plt.subplots(ndim, 1, figsize=(20, 20))
for i in range(ndim):
    for iw in range(8):
        ax[i].plot(range(nstep), chain[:, iw, i],label=iw, alpha=0.7)
        ax[i].set_ylabel(labels[i],fontsize=20)
        ax[i].set_xlim([0, nstep])
ax[0].legend()
ax[ndim-1].set_xlabel("step")
plt.tight_layout()
plt.grid(True)
plt.show()

## cornerplot

In [ ]:
import corner

flattened_chain = chain[3000:,:,:].reshape(-1,3)
flattened_chain[:, 0]=flattened_chain[:, 0]*1E-3  ## -> 1000ppm

In [ ]:

import statistics

fts=30
# Create the corner plot
fig, ax = plt.subplots(1, 3, figsize=(20, 6))

minx=[0,130,0,-1,0]
maxx=[8,200,1,1,10]

for i in range(3):
    
    # ax[i, i].hist(flattened_chain[:, i], bins=30, color="blue", alpha=0.7, density=True)
    ax[i].hist(flattened_chain[:, i], bins=30, color="blue", alpha=0.5, density=True)
    ax[i].tick_params(axis='both', labelsize=20)
    # ax[i, i].set_xlabel(labels[i],fontsize=20)
    ax[i].set_xlabel(labels[i],fontsize=fts)
    # ax[i, i].set_ylabel("PDF",fontsize=fts)
    ax[1].set_xlim([150, 200])

    counts, bins = np.histogram(flattened_chain[:, i], bins=30)

    means =np.percentile(flattened_chain[:, i], 50)
    # means =np.mean(flattened_chain[:, i])
    stdev = np.std(flattened_chain[:, i])
    # ax[i, i].axvline(means,color="b",linestyle="--",label=f"posterior: ({means:6.2f}, {stdev:5.2f})")
    if i==0:
        ax[0].axvline(means,color="b",linestyle="--",label=f"Posterior: ({means:4.0f}, {stdev:4.0f})")    
    else:
        ax[i].axvline(means,color="b",linestyle="--",label=f"Posterior: ({means:4.1f}, {stdev:4.1f})")
    

    # Plot prior distribution
    mean, stddev = [(2.5, 10), (169, 10), (1.0,0.5), (0,0.8),(5.0,1.)][i]
    x = np.linspace(minx[i], maxx[i], 300)
    prior = norm.pdf(x, mean, stddev)
    # ax[i, i].plot(x, prior, color="red", linestyle="--", label=rf"Prior ~ $N$({mean}, {stddev})")
    ax[i].plot(x, prior, color="red", linestyle="--", label=rf"Prior ({mean}, {stddev})")
    ax[i].legend(fontsize=16)


# Show the plot
plt.tight_layout()
# Show the plot
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm

# labels=[r"$xNH3$ [ppm]", r"$T_{1bar}$ [K]", r"$RH_{max}$",r"$\Delta\frac{dlnNH3}{dlnP}$",r"$P_{max}$ [bar]"]
labels=[r"$xNH3$ [1,000 ppm]", r"$T_{1bar}$ [K]", r"$RH_{max}$",r"$\Delta\Gamma$",r"$P_{max}$ [bar]"]

# Assuming flattened_chain and labels are already defined
fig, ax = plt.subplots(1,3, figsize=(18, 5))

ax = ax.flatten()
# ax[5].axis('off')

minx = [0, 169, 0.2, -0.4, 1]
maxx = [6, 186, 1, 0.2, 9]

for i in range(3):
    ax[i].hist(flattened_chain[:, i], bins=30, color="Navy", alpha=1, histtype='step', linewidth=3, density=True)
    ax[i].tick_params(axis='both', labelsize=20)
    ax[i].set_xlabel(labels[i], fontsize=25)
    # ax[0].set_xticks([330,360,390,420])
    # ax[0].set_yticks([0.,0.01,0.02,0.03])
    ax[0].set_ylabel("PDF", fontsize=25)
    # ax[3].set_ylabel("PDF", fontsize=25)
    ax[i].set_xlim(minx[i], maxx[i])

    counts, bins = np.histogram(flattened_chain[:, i], bins=30)

    # Find the peak value (the bin with the maximum count)
    peak_index = np.argmax(counts)
    peak_value = (bins[peak_index] + bins[peak_index + 1]) / 2.

    means = np.percentile(flattened_chain[:, i], 50)
    stdev = np.std(flattened_chain[:, i])

    # ax[i].axvline(means,color="navy",linestyle="--")    
    # ax[i].axvline(means-stdev,color="b",linestyle="--")    
    # ax[i].axvline(means+stdev,color="b",linestyle="--")    

    # Calculate histogram data
    counts, bin_edges = np.histogram(flattened_chain[:, i], bins=30, density=True)
  
    for ii in range(len(counts)):
        if bin_edges[ii]<=(means - stdev) and  bin_edges[ii+1]>=(means - stdev):
            ax[i].fill_between([means - stdev,bin_edges[ii+1]], 0,[counts[ii],counts[ii]],
                       color='lightgray', alpha=1., edgecolor="lightgray")
        
    for ii in range(len(counts)):
        if bin_edges[ii]<=(means + stdev) and  bin_edges[ii+1]>=(means + stdev):
            ax[i].fill_between([bin_edges[ii],means + stdev], 0,[counts[ii],counts[ii]],
                       color='lightgray', alpha=1., edgecolor="lightgray")
        
    for ii in range(len(counts)):
        if bin_edges[ii]>=(means - stdev) and  bin_edges[ii+1]<=(means + stdev):
            ax[i].fill_between([bin_edges[ii],bin_edges[ii+1]], 0,[counts[ii],counts[ii]],
                       color='lightgray', alpha=1., edgecolor="lightgray")
                
    ax[i].text(0.05, 0.95, ['a','b','c','d','e'][i], transform=ax[i].transAxes, 
                    fontsize=25, fontweight='bold', va='top', ha='left')
# Show the plot
plt.tight_layout()
plt.show()



In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm
import corner

# labels=[r"$xNH3$ [ppm]", r"$T_{1bar}$ [K]", r"$RH_{max}$",r"$\Delta\frac{dlnNH3}{dlnP}$",r"$P_{max}$ [bar]"]
labels=[r"H$_2$O", r"T$_\text{1bar}$", r"RH$_\text{max}$"]
units=[" [1,000 ppm]", " [K]",""]
unit=[r"$\times 10^3$ ppm", "K",""]

fig, ax = plt.subplots(3,3 , figsize=(18*0.6, 17*0.6))

minx = [0, 174, 0.2]
maxx = [6, 186, 1]

ticks=[[0,2,4,6],
       [175,180,185],
       [0.3,0.5,0.7,0.9]
       ]

ip=0
for i in range(3):
    for j in range(i,3):
        if i!=j:
            ax[i,j].axis('off')
            corner.hist2d(flattened_chain[:, i], flattened_chain[:, j], ax=ax[j, i], color="navy")

        ip+=1
        ax[j,i].text(0.05, 0.95, chr(96+ip), transform=ax[j,i].transAxes, fontsize=25, fontweight='bold', va='top', ha='left')

        ax[j,i].tick_params(axis='x', labelsize=18)
        ax[j,i].tick_params(axis='y', labelsize=18)

        ax[j,i].set_xlim(minx[i],maxx[i])
        if i!=j:
            ax[j,i].set_ylim(minx[j],maxx[j])

        if j==2:
            ax[j,i].set_xlabel(labels[i]+units[i], fontsize=23)
        else:
            ax[j,i].set_xlabel("")
        if i==0 and i!=j:
            ax[j,i].set_ylabel(labels[j]+units[j], fontsize=23)
        else:
            ax[j,i].set_ylabel("")            

        if i!=j:
            ax[j,i].set_yticks(ticks[j])
        ax[j,i].set_xticks(ticks[i])    

        ax[j,i].set_xticklabels([])
        ax[j,i].set_yticklabels([])
        if j==2:
            ax[j,i].set_xticklabels(ticks[i])
        if i==0 and i!=j:
            ax[j,i].set_yticklabels(ticks[j])
    ax[0,0].set_ylabel("PDF", fontsize=23)

    ## PDF for diagnal subplots
    ax[i,i].hist(flattened_chain[:, i], bins=30, color="Navy", alpha=1, histtype='step', linewidth=3, density=True)

    mmy = np.percentile(flattened_chain[:, i], 50)
    lowery = np.percentile(flattened_chain[:, i], 16)
    uppery = np.percentile(flattened_chain[:, i], 84)
    ## reflines
    ax[i,i].axvline(mmy,color="navy",linestyle="--")    
    ax[i,i].axvline(lowery,color="b",linestyle="--")    
    ax[i,i].axvline(uppery,color="b",linestyle="--")    

    # Plot prior distribution
    mean, stddev = [(2.5, 10), (169, 10), (1.0,0.5)][i]
    x = np.linspace(minx[i], maxx[i], 300)
    prior = norm.pdf(x, mean, stddev)
    ax[i,i].plot(x, prior, color="red", linestyle="-")

    ## show posteriors over head
    teex=fr"{labels[i]}=${mmy:.2f}_{{{lowery-mmy:.2f}}}^{{+{uppery-mmy:.2f}}}$ {unit[i]}"
    fig.text(0.24+i*0.92/3., 1.018-(i*0.183*5/3.), teex, ha='center', va='top', fontsize=18)


    # Calculate histogram data
    counts, bin_edges = np.histogram(flattened_chain[:, i], bins=30, density=True)

    for ii in range(len(counts)):
        if bin_edges[ii]<=(lowery) and  bin_edges[ii+1]>=(lowery):
            ax[i,i].fill_between([lowery,bin_edges[ii+1]], 0,[counts[ii],counts[ii]],
                    color='lightgray', alpha=1., edgecolor="lightgray")
        
    for ii in range(len(counts)):
        if bin_edges[ii]<=(uppery) and  bin_edges[ii+1]>=(uppery):
            ax[i,i].fill_between([bin_edges[ii],uppery], 0,[counts[ii],counts[ii]],
                    color='lightgray', alpha=1., edgecolor="lightgray")
        
    for ii in range(len(counts)):
        if bin_edges[ii]>=(lowery) and  bin_edges[ii+1]<=(uppery):
            ax[i,i].fill_between([bin_edges[ii],bin_edges[ii+1]], 0,[counts[ii],counts[ii]],
                    color='lightgray', alpha=1., edgecolor="lightgray")

# Show the plot
plt.tight_layout()
plt.savefig("Ext_MCMC_moist_corners.pdf", dpi=300,bbox_inches='tight')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import gaussian_kde
from matplotlib.colors import Normalize
from matplotlib import cm

# Fixing random state for reproducibility
# np.random.seed(19680801)

# # Some random data
# x = np.random.randn(1000)
# y = np.random.randn(1000)

x=flattened_chain[:, 0]*1E-3
y=flattened_chain[:, 1]

h5=h5py.File(f'run_juno_emcee_H2O_rk1_adapt_deeptheta_10000.h5', 'r') 
deeptheta = h5['PotentialTemperature'][3000:,:]
h5.close()
y=deeptheta.flatten()

# Calculate the 16th and 84th percentiles
mmx = np.percentile(x, 50)
lowerx = np.percentile(x, 16)
upperx = np.percentile(x, 84)

mmy = np.percentile(y, 50)
lowery = np.percentile(y, 16)
uppery = np.percentile(y, 84)

def demo_con_style(ax, text, theta1, theta2, linecolor, connectionstyle):
    x1, y1 = theta1
    x2, y2 = theta2

    # ax.plot([x1, x2], [y1, y2], ".")
    ax.annotate(text,
                xy=(x1, y1), xycoords='data',
                xytext=(x2, y2), textcoords='data',fontsize=9, 
                arrowprops=dict(arrowstyle="-", color=linecolor,linewidth=0.5,
                                shrinkA=15, shrinkB=2,
                                patchA=None, patchB=None,
                                connectionstyle=connectionstyle,
                                ),
                )

def joint_hist(x, y, ax, ax_histx, ax_histy):
    # Disable labels and ticks on the appropriate sides
    ax_histx.tick_params(axis="x", labelbottom=False, labelleft=False, left=False)
    ax_histx.tick_params(axis="y", labelleft=False, left=False)

    ax_histy.tick_params(axis="y", labelleft=False, left=True, labelright=False, right=False)
    ax_histy.tick_params(axis="x", labelbottom=False, bottom=False)
    # ax_histy.invert_xaxis()

    ax.tick_params(axis="y", labelleft=True, left=True, labelright=False, right=False)

    # ax.scatter(x, y, c='blue', s=1., alpha=0.5)

    # Create grid for contour plot
    x_grid = np.linspace(np.min(x), np.max(x), 100)
    y_grid = np.linspace(np.min(y), np.max(y), 100)
    X, Y = np.meshgrid(x_grid, y_grid)
    Z = gaussian_kde(np.vstack([x, y]))(np.vstack([X.ravel(), Y.ravel()])).reshape(X.shape)

    # Create a custom colormap using the first half of 'bwr_r'
    original_cmap = cm.get_cmap('bwr')
    custom_cmap = original_cmap(np.linspace(0.5, 0, 256))  # Extract the first half
    custom_cmap = cm.colors.ListedColormap(custom_cmap)

    # Plot contours using the custom colormap
    cont = ax.contourf(X, Y, Z, levels=100, cmap=custom_cmap, alpha=1)

    X, Y = np.meshgrid(x_grid, y_grid)
    kde = gaussian_kde([x, y])
    mean = [np.mean(x), np.mean(y)]
    cov = np.cov(x, y)

    from scipy.stats import multivariate_normal
    levels = [multivariate_normal.pdf(mean, mean=mean, cov=cov) * 0.6065306597,  # ~1 std dev
            multivariate_normal.pdf(mean, mean=mean, cov=cov) * 0.1353352832,  # ~2 std dev
            multivariate_normal.pdf(mean, mean=mean, cov=cov) * 0.011, # ~3 std dev
            ] 
    Z = kde(np.vstack([X.ravel(), Y.ravel()])).reshape(X.shape)
    CS = ax.contour(X, Y,Z, levels=[levels[0],],  colors=('b',), linewidths=1, alpha=1)
    CS = ax.contour(X, Y,Z, levels=[levels[1],],  colors=('b',), linewidths=1, alpha=0.5)
    CS = ax.contour(X, Y,Z, levels=[levels[2],],  colors=('b',), linewidths=1, alpha=0.3)
    # CS = ax.contour(X, Y,Z, levels=levels[::-1],  colors=((1, 0, 0,1), (1, 0, 0,0.7), (1, 0, 0, 0.5)), linewidths=2)


    # demo_con_style(ax, "39.3%",  [3,178.4], [3.2,181], "k","arc,angleA=0,angleB=-90,armA=120,rad=0")
    # demo_con_style(ax, "86.5%", [3,178], [3.2,179], "blue","arc,angleA=0,angleB=-90,armA=120,rad=0")
    ax.text(3.1,177.3, "39.3%",fontsize=9)
    ax.text(4,178.5, "86.5%",fontsize=9)
    ax.text(4.5,179.7, "98.9%",fontsize=9)



    ax_histx.hist(x, bins=30, density=True, color="Navy", alpha=1, histtype='step')
    ax_histy.hist(y, bins=30, density=True, color='Navy', alpha=1, histtype='step', orientation='horizontal')

    # ax_histy.set_xlim(0.22,0)


    means = np.mean(x)
    stdev = np.std(x)
    counts, bin_edges = np.histogram(x, bins=30, density=True)
  
    for ii in range(len(counts)):
        if bin_edges[ii]<=(means - stdev) and  bin_edges[ii+1]>=(means - stdev):
            ax_histx.fill_between([means - stdev,bin_edges[ii+1]], 0,[counts[ii],counts[ii]],  color='blue', alpha=0.5,linewidth=0)
        
    for ii in range(len(counts)):
        if bin_edges[ii]<=(means + stdev) and  bin_edges[ii+1]>=(means + stdev):
            ax_histx.fill_between([bin_edges[ii],means + stdev], 0,[counts[ii],counts[ii]],  color='blue', alpha=0.5,linewidth=0)
        
    for ii in range(len(counts)):
        if bin_edges[ii]>=(means - stdev) and  bin_edges[ii+1]<=(means + stdev):
            ax_histx.fill_between([bin_edges[ii],bin_edges[ii+1]], 0,[counts[ii],counts[ii]],  color='blue', alpha=0.5,linewidth=0)

    means = np.mean(y)
    stdev = np.std(y)
    counts, bin_edges = np.histogram(y, bins=30, density=True)
  
    for ii in range(len(counts)):
        if bin_edges[ii]<=(means - stdev) and  bin_edges[ii+1]>=(means - stdev):
            ax_histy.fill_between([counts[ii],0],means - stdev,bin_edges[ii+1], color='blue', alpha=0.5,linewidth=0)
        
    for ii in range(len(counts)):
        if bin_edges[ii]<=(means + stdev) and  bin_edges[ii+1]>=(means + stdev):
            ax_histy.fill_between([counts[ii],0],bin_edges[ii],means + stdev, color='blue', alpha=0.5,linewidth=0)
        
    for ii in range(len(counts)):
        if bin_edges[ii]>=(means - stdev) and  bin_edges[ii+1]<=(means + stdev):
            ax_histy.fill_between([counts[ii],0],bin_edges[ii],bin_edges[ii+1], color='blue', alpha=0.5,linewidth=0)


# Create a figure with the constrained layout
fig = plt.figure(figsize=(14,14),dpi=300)
# fig = plt.figure(layout='constrained')

# Create the main scatter plot axes
ax = fig.add_gridspec(top=0.75, left=0.75).subplots()
# ax.set(aspect=1)
ax.set(aspect=6/15) 
# Create marginal histograms for x and y axes
ax_histx = ax.inset_axes([0, 1.05, 1, 0.25], sharex=ax)
ax_histy = ax.inset_axes([1.05, 0, 0.25, 1], sharey=ax)

ax.set_xlim(0,6)
ax.set_ylim(170,185)

ax.set_yticks([172,176,180,184])
# ax.set_xticks([340,360,380,400,420])

ax.set_xlabel(r'$\text{H}_2$O [1,000 ppm]')
ax.set_ylabel(r'$\theta$ [K]')
ax.yaxis.set_label_position("left")

# ax_histx.axvline(x=lowerx, color='black', linestyle='--')
# ax_histx.axvline(x=upperx, color='black', linestyle='--')

# ax_histy.axhline(y=lowery, color='black', linestyle='--')
# ax_histy.axhline(y=uppery, color='black', linestyle='--')

ax_histx.set_xlabel(rf"H$_2$O$={mmx:.1f}_{{{lowerx-mmx:.1f}}}^{{+{upperx-mmx:.1f}}} \times 10^3$ ppm")
ax_histy.set_ylabel(rf"$\theta={mmy:.2f}_{{{lowery-mmy:.2f}}}^{{+{uppery-mmy:.2f}}}$ K", rotation=-90,labelpad=13)

ax_histx.xaxis.set_label_position("top")
ax_histy.yaxis.set_label_position("right")


## crossing
# ax.axhline(y=mmy, linestyle="--",color='red', linewidth=1)
# ax.axvline(x=mmx, linestyle="--",color='blue', linewidth=1)
# ax_histy.axhline(y=mmy,linestyle="--", color='red', linewidth=1)
# ax_histx.axvline(x=mmx,linestyle="--", color='blue', linewidth=1)


# Draw the density plot and marginal histograms
joint_hist(x, y, ax, ax_histx, ax_histy)

fig.text(0.71, 0.56, "b", fontsize=16, fontweight='bold', va='top', ha='left')

plt.show()


In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt

# Open Juno MWR NetCDF file
juno = h5py.File("./juno_mwr-main.nc", "r")

# Read pressure data and convert units
pressure_bars = juno["press"][0,:,0,0] * 0.00001
# print("Pressure Summary:")
# print(f"Min: {np.min(press)}, Max: {np.max(press)}, Mean: {np.mean(press)}, Std: {np.std(press)}")

# Open profile HDF5 file
profile = h5py.File("../../build_euler_apative_bg/bin/run_regenerate_dry_bg_Temps_profile_5000.h5", "r")

# Read Temperature data (assuming it's Temperature, adjust if it's PotentialTemperature)
rh = profile["Temperature"][1000:, 0, :]

# Calculate mean and standard deviation along the first dimension
tempNH3 = np.mean(rh, axis=0)
devNH3 = np.std(rh, axis=0)

# Open another profile HDF5 file (assuming this is a different file)
profile2 = h5py.File("run_juno_emcee_H2O_rk1_adapt_Temps_profile_10000.h5", "r")

# Read Temperature data (assuming it's Temperature, adjust if it's PotentialTemperature)
rh2 = profile2["Temperature"][3000:, 0, :]

# Calculate mean and standard deviation along the first dimension
tempH2O = np.mean(rh2, axis=0)
devH2O = np.std(rh2, axis=0)

# Print the minimum and maximum values of devH2O
print("devH2O Min-Max:")
print(f"Min: {np.min(devH2O)}, Max: {np.max(devH2O)}")

# Generate 100 random indices between 0 and 99
random_indices = np.random.randint(0, len(rh), size=100)

# print(random_indices)


fig, ax1 = plt.subplots(figsize=(3.5, 5),dpi=300)

ax1.plot(tempNH3, pressure_bars, label='Dry adiabatic', color='red', alpha=1, linewidth=1)
ax1.plot(tempH2O, pressure_bars, label='Moist adiabatic', color='blue', alpha=1, linewidth=1)
ax1.plot([-1,0], [1E8,1E8], label='Moist - Dry', color='k', linewidth=1)
# Label the primary axes
ax1.set_xlabel('Temperature [K]')
ax1.set_ylabel('Pressure [bar]')  # Units in bar
# ax1.set_title('Temperature Profiles of NH3 and H2O')
ax1.legend(frameon=False, fontsize=8)
# ax1.grid()

# Set y-axis to logarithmic scale
ax1.set_yscale('log')
ax1.set_xscale('log')

# Set y-limits
ax1.set_ylim(0.2, 30)  # Inverted to match log scale
ax1.set_xlim(100,500)  # Inverted to match log scale
ax1.invert_yaxis()  # Invert y-axis for atmospheric profiles

ax1.set_xticks([100,200,300,400,500])
ax1.set_xticklabels([100,200,300,400,500])

from matplotlib.ticker import FixedLocator
ax1.minorticks_on()
ax1.yaxis.set_minor_locator(FixedLocator([0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1,2,3,4,5,6,7,8,9,10,20,30]))
ax1.set_yticks([0.2,1,10,30])
ax1.set_yticklabels([0.2,1.0,10,30])


# Calculate the difference between tempH2O and tempNH3
temp_difference = tempH2O- tempNH3


ax2 = ax1.twiny()
# ax2.axvline(0,color="k", linestyle="--", alpha=0.5, linewidth=0.5 )
ax2.plot(temp_difference, pressure_bars, label='Moist - Dry', color='k', linewidth=1)
ax2.fill_betweenx(pressure_bars, temp_difference - devH2O, temp_difference + devH2O, color='gray', alpha=0.2, linewidth=0)
ax2.set_xlim(-5,10)
ax2.set_xticks([-5,-2,0,2,5,10])
ax2.tick_params(axis='x', pad=1)  # Padding for x-axis tick labels
# Label the secondary axes


ax2.set_xlabel('Difference [K]', labelpad=5)

axtau=ax1.twiny()
df=h5py.File("/home/jihenghu/JHCanoe/build_euler_apative_bg/bin/juno_mwr_rt_fwd-main.nc","r")
tau1=np.array(df["CH1-tau"][0,:,0,0])
tau2=np.array(df["CH2-tau"][0,:,0,0])
tau3=np.array(df["CH3-tau"][0,:,0,0])
tau4=np.array(df["CH4-tau"][0,:,0,0])
tau5=np.array(df["CH5-tau"][0,:,0,0])
tau6=np.array(df["CH6-tau"][0,:,0,0])
temp=np.array(df["temp"][0,:,0,0])
z=np.array(df["x1"])

w1=[0]*1600
w2=[0]*1600
w3=[0]*1600
w4=[0]*1600
w5=[0]*1600
w6=[0]*1600

for i in np.arange(len(temp)-1-1,-1,-1):
    # print(i)
    tau1[i]= tau1[i]+tau1[i+1]
    tau2[i]= tau2[i]+tau2[i+1]
    tau3[i]= tau3[i]+tau3[i+1]
    tau4[i]= tau4[i]+tau4[i+1]
    tau5[i]= tau5[i]+tau5[i+1]
    tau6[i]= tau6[i]+tau6[i+1]


# from top to bottom  
for i in np.arange(len(temp)-1-1,-1,-1):
    TeTau1=( temp[i]*np.exp(-1*tau1[i]) +  temp[i-1]*np.exp(-1*tau1[i+1]) )*0.5
    TeTau2=( temp[i]*np.exp(-1*tau2[i]) +  temp[i-1]*np.exp(-1*tau2[i+1]) )*0.5
    TeTau3=( temp[i]*np.exp(-1*tau3[i]) +  temp[i-1]*np.exp(-1*tau3[i+1]) )*0.5
    TeTau4=( temp[i]*np.exp(-1*tau4[i]) +  temp[i-1]*np.exp(-1*tau4[i+1]) )*0.5
    TeTau5=( temp[i]*np.exp(-1*tau5[i]) +  temp[i-1]*np.exp(-1*tau5[i+1]) )*0.5
    TeTau6=( temp[i]*np.exp(-1*tau6[i]) +  temp[i-1]*np.exp(-1*tau6[i+1]) )*0.5

    dtau1= tau1[i]-tau1[i+1]
    dtau2= tau2[i]-tau2[i+1]
    dtau3= tau3[i]-tau3[i+1]
    dtau4= tau4[i]-tau4[i+1]
    dtau5= tau5[i]-tau5[i+1]
    dtau6= tau6[i]-tau6[i+1]

    w1[i]=TeTau1*dtau1/(z[i+1]-z[i])/temp[i]
    w2[i]=TeTau2*dtau2/(z[i+1]-z[i])/temp[i]
    w3[i]=TeTau3*dtau3/(z[i+1]-z[i])/temp[i]
    w4[i]=TeTau4*dtau4/(z[i+1]-z[i])/temp[i]
    w5[i]=TeTau5*dtau5/(z[i+1]-z[i])/temp[i]
    w6[i]=TeTau6*dtau6/(z[i+1]-z[i])/temp[i]

w1=w1/max(w1)
w2=w2/max(w2)
w3=w3/max(w3)
w4=w4/max(w4)
w5=w5/max(w5)
w6=w6/max(w6)
axtau.plot(w1, pressure_bars, "k--",alpha=0.3, linewidth=1)
axtau.plot(w2, pressure_bars, "k--",alpha=0.3, linewidth=1)
axtau.plot(w3, pressure_bars, "k--",alpha=0.3, linewidth=1)
axtau.plot(w4, pressure_bars, "k--",alpha=0.3, linewidth=1)
axtau.plot(w5, pressure_bars, "k--",alpha=0.3, linewidth=1)
axtau.plot(w6, pressure_bars, "k--",alpha=0.3, linewidth=1)
axtau.get_xaxis().set_visible(False)
fig.text(-0.02, 0.96, "c", fontsize=16, fontweight='bold', va='top', ha='left')

plt.show()

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt


# Open Juno MWR NetCDF file
juno = h5py.File("./juno_mwr-main.nc", "r")

# Read pressure data and convert units
pressure_bars = juno["press"][0,:,0,0] * 0.00001

# Specify the filename
filename = "run_juno_emcee_H2O_rk1_adapt_H2O_NH3_molefrac_10000step_last5000.h5"
# filename = "run_juno_emcee_H2O_rk1_adapt_H2O_NH3_mixingratio_10000step_last5000.h5"

# Open the HDF5 file in read mode
with h5py.File(filename, 'r') as profIN:
    # Extract datasets
    SH_NH3 = profIN['SH_NH3'][:, :, :]*1E6
    SH_H2O = profIN['SH_H2O'][:, :, :]*1E6/1000.



filename = "../../build_euler_apative_bg/bin/New_run_juno_emcee_dry_NH3_T_RHmax_adlnNH3_parallel_pmaxstd1_bg_NH3_H2O_molefrac_step10000_last5000.h5"
# filename = "../../build_euler_apative_bg/bin/New_run_juno_emcee_dry_NH3_T_RHmax_adlnNH3_parallel_pmaxstd1_bg_NH3_H2O_mixingratio_step10000_last5000.h5"

with h5py.File(filename, 'r') as profIN:
    # Extract datasets
    SH_NH3_dry = profIN['SH_NH3'][:, :, :]*1E6
    SH_H2O_dry = profIN['SH_H2O'][:, :, :]*1E6/1000.



H2O = np.median(SH_H2O, axis=(0,1))
devH2O = np.std(SH_H2O, axis=(0,1))

NH3 = np.median(SH_NH3, axis=(0,1))
devNH3 = np.std(SH_NH3, axis=(0,1))


H2O_dry = np.median(SH_H2O_dry, axis=(0,1))
devH2O_dry = np.std(SH_H2O_dry, axis=(0,1))

NH3_dry = np.median(SH_NH3_dry, axis=(0,1))
devNH3_dry = np.std(SH_NH3_dry, axis=(0,1))


fig, ax1 = plt.subplots(figsize=(3.5,5),sharey=True, dpi=300)
# ax1=axes[0]
# Plot tempNH3 with standard deviation
ax1.plot(100000,1E-9,'k-', label=r'Dry adiabatic')
ax1.plot(100000,1E-9,'k--', label=r'Moist adiabatic')

# ax1.fill_betweenx(pressure_bars, H2O - devH2O, H2O + devH2O, color='blue', alpha=0.2, linewidth=0)


ax1.plot(H2O, pressure_bars,'k--')
ax1.fill_betweenx(pressure_bars, H2O - devH2O, H2O + devH2O, color='blue', alpha=0.3, linewidth=0)

ax1.plot(H2O_dry, pressure_bars, 'k-')
ax1.fill_betweenx(pressure_bars, H2O_dry - devH2O_dry, H2O_dry + devH2O_dry, color='b', alpha=0.3, linewidth=0)


ax1.set_ylabel('Pressure [bar]')  # Units in bar
# ax1.set_title('Temperature Profiles of NH3 and H2O')
ax1.legend(frameon=False)
# ax1.grid()
ax1.set_xlabel(r'H$_2$O [1,000 ppm]', color="blue")
# Set y-axis to logarithmic scale
ax1.set_yscale('log')
# ax1.set_xscale('log')


from matplotlib.ticker import FixedLocator
ax1.minorticks_on()
ax1.yaxis.set_minor_locator(FixedLocator([0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1,2,3,4,5,6,7,8,9,10,20,30]))
ax1.set_yticks([0.2,1,10,30])
ax1.set_yticklabels([0.2,1.0,10,30])


# Set y-limits
ax1.set_ylim(0.2, 30)  # Inverted to match log scale
ax1.set_xlim(0,4)  # Inverted to match log scale
# ax1.set_xlim(0,0.025)  # Inverted to match log scale
ax1.invert_yaxis()  # Invert y-axis for atmospheric profiles


ax2=ax1.twiny()
ax2.plot(NH3, pressure_bars,'k--')
ax2.fill_betweenx(pressure_bars, NH3 - devNH3, NH3 + devNH3, color='green', alpha=0.3, linewidth=0)

ax2.plot(NH3_dry, pressure_bars, 'k-')
ax2.fill_betweenx(pressure_bars, NH3_dry - devNH3_dry, NH3_dry + devNH3_dry, color='green', alpha=0.3, linewidth=0)

ax2.set_xlabel(r'NH$_3$ [ppm]', color="green")
# Set y-limits
ax2.set_ylim(0.2, 30)  # Inverted to match log scale
ax2.set_xlim(0,420)  # Inverted to match log scale
ax2.invert_yaxis()  # Invert y-axis for atmospheric profiles

plt.tight_layout()
fig.text(0.02, 0.96, "d", fontsize=16, fontweight='bold', va='top', ha='left')

plt.show()

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt


# Open Juno MWR NetCDF file
juno = h5py.File("./juno_mwr-main.nc", "r")

# Read pressure data and convert units
pressure_bars = juno["press"][0,:,0,0] * 0.00001


# filename = "../../build_euler_apative_bg/bin/New_run_juno_emcee_dry_NH3_T_RHmax_adlnNH3_parallel_pmaxstd1_bg_NH3_H2O_molefrac_step10000_last5000.h5"
filename = "/home/jihenghu/JHCanoe/build_bg_cpcs/bin/run_regenerate_Tb_profile_parallel_bg_NH3_5000.h5"
with h5py.File(filename, 'r') as profIN:
    # Extract datasets
    SH_NH3_dry = profIN['SH_NH3'][1000:, :, :]#*1E6



NH3_dry = np.median(SH_NH3_dry, axis=(0,1))
devNH3_dry = np.std(SH_NH3_dry, axis=(0,1))


fig, ax1 = plt.subplots(figsize=(4, 6),dpi=300)



ax1.plot(NH3_dry, pressure_bars, '-', color='Purple', label=r'NH$_3$ dry adiabatic')
ax1.fill_betweenx(pressure_bars, NH3_dry - devNH3_dry, NH3_dry + devNH3_dry, color='Purple', alpha=0.2, linewidth=0)


# ax1.plot(-1000,-10000,linestyle="--", color="k", label="dry adiabate")
# ax1.plot(-1000,-10000,linestyle="-", color="k", label="moist adiabate")

# Label the primary axes
ax1.set_xlabel(r'Mixing ratio [ppm]')
ax1.set_ylabel('Pressure [bar]')  # Units in bar
# ax1.set_title('Temperature Profiles of NH3 and H2O')
ax1.legend()
# ax1.grid()

# Set y-axis to logarithmic scale
ax1.set_yscale('log')
# ax1.set_xscale('log')

# Set y-limits
# ax1.set_ylim(0.2, 30)  # Inverted to match log scale
# ax1.set_xlim(1E-3,1E4)  # Inverted to match log scale
ax1.invert_yaxis()  # Invert y-axis for atmospheric profiles

from matplotlib.ticker import FixedLocator
# ax1.minorticks_on()
# ax1.yaxis.set_minor_locator(FixedLocator([0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1,2,3,4,5,6,7,8,9,10,20,30]))
# ax1.set_yticks([0.2,1,10,30])
# ax1.set_yticklabels([0.2,1.0,10.0,30.0])



plt.show()